# Plant Health Monitoring using Deep Learning
Final 3-class instance segmentation project: healthy leaf, unhealthy leaf, and stem.


## 1. Setup


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
print(os.getcwd())


## 2. Dataset Validation
Dataset must be COCO format and split 80/10/10.


In [ ]:
!python src/validate_coco.py --annotations data/annotations/train.json --image_dir data/images/train
!python src/validate_coco.py --annotations data/annotations/val.json --image_dir data/images/val
!python src/validate_coco.py --annotations data/annotations/test.json --image_dir data/images/test


## 3. Baseline Before Fine-tuning
Evaluate the adapted pretrained model before plant-data training.


In [ ]:
!python src/evaluate_baseline_no_training.py --config configs/maskrcnn_config.yaml --split val --threshold 0.05


## 4. Fine-tuning with Augmentation
Head-only training is followed by full fine-tuning.


In [ ]:
!python src/train_maskrcnn.py --config configs/maskrcnn_config.yaml


## 5. Hyperparameter Random Search


In [ ]:
!python src/random_search_maskrcnn.py --config configs/maskrcnn_config.yaml --max_trials 4 --epochs 5


## 6. Evaluation on Test Set


In [ ]:
!python src/evaluate_maskrcnn.py --config configs/maskrcnn_config.yaml --weights outputs/checkpoints/maskrcnn_best.pth --split test --threshold 0.05


## 7. Recent Technique: NMS, Top-k Filtering, Threshold Tuning, and TTA


In [ ]:
# Replace YOUR_IMAGE.jpg with one image from data/images/test
!python src/predict_maskrcnn.py --image data/images/test/YOUR_IMAGE.jpg --weights outputs/checkpoints/maskrcnn_best.pth --threshold 0.05 --top_k 6 --tta


## 8. Visualizations


In [ ]:
!python src/full_visualizations.py
from IPython.display import display
for f in ['outputs/visualizations/training_validation_loss.png','outputs/visualizations/hyperparameter_tuning.png','outputs/visualizations/mean_iou_comparison.png','outputs/visualizations/mean_dice_comparison.png','outputs/visualizations/threshold_tradeoff.png']:
    if os.path.exists(f):
        display(Image.open(f))
    else:
        print('Missing:', f)


## 9. Mini-network from Scratch
Mini U-Net baseline for Part 2. Requires semantic masks converted from COCO polygons.


In [ ]:
!python src/coco_to_semantic_masks.py --annotations data/annotations/train.json --output_dir data/semantic_masks/train
!python src/coco_to_semantic_masks.py --annotations data/annotations/val.json --output_dir data/semantic_masks/val
!python src/train_unet.py --config configs/unet_config.yaml
